# Learning module 5 — Use a project-local Node runtime

**Small milestone:** prepare the Node prerequisite, activate it for the notebook, and verify the executable and versions.

We installed the official **Node 24.22.2 macOS arm64** archive under `server/.tools/node-v24.22.2-darwin-arm64`. The archive's SHA-256 matched the published manifest before extraction. Node 24 is an LTS release line and satisfies the documented Appium Node requirement. [Node release status](https://nodejs.org/en/about/previous-releases), [Appium requirements](https://appium.io/docs/en/latest/quickstart/requirements/)

| Artifact | Purpose |
| --- | --- |
| `server/.node-version` | Records the exact selected version: `24.22.2`. It does not activate Node by itself. |
| `server/.tools/node-v24.22.2-darwin-arm64/` | Holds the local runtime, including bundled npm. |
| `server/.gitignore` | Excludes the downloaded runtime directory from Git. |

Installation source: [official archive](https://nodejs.org/dist/v24.22.2/node-v24.22.2-darwin-arm64.tar.gz). The matching entry came from the [official Node 24 checksum manifest](https://nodejs.org/dist/latest-v24.x/SHASUMS256.txt); its SHA-256 was `aa9549349e47dffaa42dcf95d6e06ade8c4bdafc980979883b88767670df385a`.

Verification during this step found Node `v24.22.2` and bundled npm `11.19.1`. The existing `/usr/local/bin/node` remained `v20.17.0`.

**Installation and activation are different.** The next cell puts this runtime's `bin` directory first on the notebook kernel's PATH. Programs started by that kernel will then find this Node/npm. Other terminals keep their own PATH. Restarting the kernel requires running the activation cell again.

Run from the project's root or the `server` folder. This changes neither the selected Python interpreter nor shell startup files. The current kernel should remain the project's Python `.venv`. Appium installation is a later milestone.

## This notebook

Part **05 — Activate the project-local Node runtime** of the learning sequence. [Notebook index](README.md).

Select the project's `.venv` Python kernel and run this notebook from top to bottom. Its Python setup is included locally; running another notebook in another kernel does not populate this one's variables or PATH. The supported working directories are the project root, `server/`, and `server/sample_scenarios/`.

**Execution scope:** Kernel PATH plus Node/npm version checks.

Saved lesson outputs and embedded screenshots are historical snapshots preserved during the split on November 3, 2026. Newly included setup cells have no saved outputs. Rerunning an executable scenario creates fresh local artifacts; these continue living in `server/.tools/`, not beside the notebooks.


In [9]:
import os
from pathlib import Path
import shutil
import subprocess

notebook_cwd = Path.cwd().resolve()
server_candidates = (notebook_cwd, notebook_cwd / "server", *notebook_cwd.parents)
server_dir = next((candidate for candidate in server_candidates
                   if (candidate / ".node-version").is_file()
                   and (candidate / "package.json").is_file()
                   and (candidate / "sample_scenarios").is_dir()), None)
if server_dir is None:
    raise RuntimeError("Run from the project root, server, or sample_scenarios folder.")

node_version = (server_dir / ".node-version").read_text().strip()
node_bin = server_dir / ".tools" / f"node-v{node_version}-darwin-arm64" / "bin"
if not (node_bin / "node").is_file():
    raise FileNotFoundError(f"Project-local Node is missing: {node_bin}")

path_entries = [entry for entry in os.environ.get("PATH", "").split(os.pathsep)
                if entry != str(node_bin)]
os.environ["PATH"] = os.pathsep.join([str(node_bin), *path_entries])

print("Selected Node:", shutil.which("node"))
print("Node version:", subprocess.check_output(["node", "--version"], text=True, timeout=10).strip())
print("npm version:", subprocess.check_output(["npm", "--version"], text=True, timeout=10).strip())


Selected Node: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/node-v24.22.2-darwin-arm64/bin/node
Node version: v24.22.2
npm version: 11.19.1


### Read the selected runtime, then pause

The first printed path should point inside this project's `server/.tools/` directory. The versions should be Node `v24.22.2` and npm `11.19.1` for this recorded installation.

`PATH` is the ordered list of locations searched for commands. Prepending our local `bin` makes this kernel's `node` and `npm` resolve there. Removing an existing identical entry first keeps repeated runs from accumulating duplicate entries.

Python continues running the notebook. Node will run the Appium server later. Selecting a Node runtime does not install Appium, its XCUITest driver, or the Python client.

A fresh [readiness notebook](04_environment_readiness.ipynb) starts with its own kernel environment. To compare against this selection, run Node activation in that same kernel first; opening another notebook does not transfer this kernel's PATH. Saved outputs remain historical snapshots until rerun.

**Pause here.** Trace the version pin → local runtime directory → kernel PATH → selected executable. Our next small milestone will be choosing and installing a compatible Appium server version into the local tooling setup.

[Index](README.md) · [Previous lesson](04_environment_readiness.ipynb) · [Next lesson](06_appium_server_package.ipynb)